In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time_series.models import KernelRidgeRegression, EigenGausRascutti, RascuttiModel
from time_series.data_handlers import TimeSeriesData
from time import time

2026-03-05 11:48:03.960 | INFO     | time_series.config:<module>:13 - PROJ_ROOT path is: /home/james/Repo/PhD Repo/time_series_clustering


In [2]:
def dynamics_sincos(
        theta: float,
        n_correlated_dimensions: int,
        n_uncorrelated_dimensions: int,
    ):

    n_dim = n_correlated_dimensions + n_uncorrelated_dimensions

    def f(x):
        # print(x.shape)
        x_next = np.zeros_like(x)

        for i in range(n_correlated_dimensions):
            x_next[i] += np.cos(theta * x[i]) 
            if i > 0:
                x_next[i] += -np.sin(theta*x[i-1])

            if i + 1 < n_dim:
                x_next[i] += np.sin(theta*x[i+1])

        # ------------------------------
        # Uncorrelated dimensions
        # ------------------------------
        for i in range(n_correlated_dimensions, n_dim):
            x_next[i] = np.cos(theta * x[i])

        return x_next
    return f

def time_series_generator(
    dynamics,
    x0: np.array,
    n_points: int,
    noise: float = 0.0,
):
    if n_points <= 0:
        raise ValueError("n_points must be positive")

    rng = np.random.default_rng()
    n_dim = len(x0)

    X = np.empty((n_points + 1, n_dim))
    X[0] = x0

    for t in range(n_points):
        x = X[t]
        x_next = dynamics(x)

        x_next += rng.normal(0.0, noise, size=n_dim)

        X[t + 1] = x_next

    return X


def create_dataset(
    theta: float,
    n_points: int,
    n_correlated_dimensions: int,
    n_uncorrelated_dimensions: int,
    noise: float = 0.0,
    seed: int | None = None,
):
    """
    Generate a bounded nonlinear dynamical system dataset.

    """

    if n_points <= 0:
        raise ValueError("n_points must be positive")

    rng = np.random.default_rng()

    n_dim = n_correlated_dimensions + n_uncorrelated_dimensions
    X = np.empty((n_points + 1, n_dim))
    X[0] = rng.uniform(-1.0, 1.0, size=n_dim)

    for t in range(n_points):
        x = X[t]
        x_next = np.zeros_like(x)

        # ------------------------------
        # Correlated nonlinear dynamics
        # ------------------------------
        for i in range(n_correlated_dimensions):
            x_next[i] += np.cos(theta * x[i]) 
            if i > 0:
                x_next[i] += -np.sin(theta*x[i-1])

            if i + 1 < n_dim:
                x_next[i] += np.sin(theta*x[i+1])

        # ------------------------------
        # Uncorrelated dimensions
        # ------------------------------
        for i in range(n_correlated_dimensions, n_dim):
            x_next[i] = np.cos(theta * x[i])

        # ------------------------------
        # Damping + noise (bounded step)
        # ------------------------------
        x_next += rng.normal(0.0, noise, size=n_dim)

        X[t + 1] = x_next

    return X


In [3]:
dynamics = dynamics_sincos(
    np.pi/4, 3, 3
)

X = time_series_generator(
    dynamics, 
    np.random.normal(0, 1, 6),
    500,
    0.2
)

dataset = TimeSeriesData(
    X[:-1, :],
    X[1:, :],
    lag=1,
    train_val_test_split=[0.5, 0.3, 0.2]
)

In [4]:
X_train, y_train = dataset.train_data()
X_val, y_val = dataset.val_data()
X_test, y_test = dataset.test_data()

In [5]:
t1 = time()

krr_model = KernelRidgeRegression(
    kernel = "rbf",
    reg = 1e-9,
    bandwidth = 1
)

krr_model.fit(X_train, y_train)

score = np.mean((y_val - krr_model.predict(X_val)))
t2 = time()

print(
f"""
Model: KRR
Score: {score:.3f},
Time: {t2 - t1}
"""
)


Model: KRR
Score: 0.061,
Time: 0.19385814666748047



In [6]:
# t1 = time()

# rascutti_model = RascuttiModel(
#     kernel = "rbf",
#     bandwidth = 1
# )

# rascutti_model.fit(X_train, y_train)

# score = np.mean((y_val - rascutti_model.predict(X_val)))
# t2 = time()

# print(
# f"""
# Model: Rascutti
# Score: {score:.3f},
# Time: {t2 - t1}
# """
# )

In [7]:
t1 = time()

eigen_rascutti_model = EigenGausRascutti(
    bandwidth = 1
)

eigen_rascutti_model.fit(X_train, y_train)

score = np.mean((y_val - eigen_rascutti_model.predict(X_val)))
t2 = time()

print(
f"""
Model: Rascutti
Score: {score:.3f},
Time: {t2 - t1}
"""
)

ValueError: Incompatible dimensions (250, 6) (10, 1)